# Deep-Dive Conceptual Roadmap & Dataset Ecosystem

Supervised Fine-Tuning (SFT) is the critical transformative step that shifts a foundational Large Language Model (LLM) from a stochastic text-completion engine into an interactive, task-oriented assistant.

---

## Core Terminology & Mechanics

- **Causal Language Modeling (CLM):** The underlying auto-regressive mechanism where the model predicts the probability distribution of the next token based on the preceding context sequence.
- **Teacher Forcing:** A training paradigm where the model is fed the ground-truth sequence up to step $t$ to predict the token at step $t+1$, regardless of what the model might have predicted during inference.
- **Categorical Cross-Entropy Loss:** The loss function used to measure the divergence between the model's predicted probability distribution and the actual one-hot encoded ground truth token.

### The Crisp Definition

SFT aligns a pre-trained base model to human-desired interaction formats by utilizing Teacher Forcing to optimize auto-regressive next-token prediction probabilities over a curated dataset of instruction-response pairs, penalizing incorrect generation via Cross-Entropy Loss.

### The Engineering Problem Solved

Base models only know how to mimic the distribution of internet data. SFT solves the **"alignment penalty"** by mapping broad web-text priors to a narrow, highly structured input/output manifold, enabling zero-shot task execution (e.g., Q&A, summarization, coding) without requiring few-shot prompting.


## The 'Why': Architectural & Mathematical Rationale

In SFT, the objective is to minimize the **negative log-likelihood** of the target sequence. The Cross-Entropy Loss formula applied is:

$$L_{\text{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \log P(x_i \mid x_{<i}, \Theta)$$

where:
- $N$ is the sequence length
- $x_i$ is the target token
- $x_{<i}$ is the context
- $\Theta$ represents the model weights

Crucially, modern SFT pipelines implement **Prompt Masking** (or loss masking). The loss is only computed on the tokens belonging to the assistant's response. The instruction/prompt tokens are masked out (typically by setting their labels to `-100` in PyTorch), ensuring the model optimizes only for **answering**, not for predicting the user's prompt.

## VRAM & Compute Impact

### Baseline (Full Fine-Tuning)

Updating a 7B parameter model in FP16/BF16 requires holding:
- Weights → **14 GB**
- Gradients → **14 GB**
- AdamW optimizer states → **28 GB**
- Forward activations → additional overhead

**Total requirement: >70 GB VRAM per GPU**, necessitating multi-node setups (FSDP/DeepSpeed).

### PEFT / QLoRA Scaling

By freezing the base model in **4-bit NormalFloat (NF4)** and training low-rank adapter matrices (LoRA):
- VRAM requirements plummet dramatically
- A 7B model can be fine-tuned on a **single 24 GB VRAM consumer GPU**
- Compute is marginally slower due to on-the-fly dequantization
- Scaling is drastically democratized

## Architectural Trade-Offs

### ✅ Pros

- **Behavioral Injection:** Highly effective at teaching the model specific tones, JSON output structures, or tool-calling schemas.
- **Efficiency via QLoRA:** Can be integrated into CI/CD pipelines for continuous domain adaptation without massive infrastructure.

### ❌ Cons

- **Catastrophic Forgetting:** Over-training on a narrow dataset can cause the model to forget broad knowledge acquired during pre-training.
- **Hallucination Risk:** SFT teaches the model to *sound confident*. If trained on data where it lacks underlying pre-trained knowledge, it will smoothly hallucinate answers.

# Production-Grade Code / Configuration

Below is a complete, production-ready script utilizing modern `trl` and `peft` APIs to execute **QLoRA-based SFT**. It uses the `no_robots` dataset and enforces strict hardware efficiency flags.

## Environment Setup

In [ ]:
# Uninstall existing torch, torchvision, and torchaudio (especially if they are CPU versions)
# !pip uninstall torch -y

# Install CUDA-enabled PyTorch, torchvision, and torchaudio
# (adjust cu version if needed, e.g., cu121, cu118 - currently using cu121)
# %pip install torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch

# Install only the necessary libraries for SFT and QLoRA on T4
# We avoid flash-attn and use the native SDPA instead.
%pip install trl bitsandbytes peft accelerate transformers

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from trl import SFTTrainer, SFTConfig

In [ ]:
# ---
# Hardware & Precision Configuration
# ---
# Enforce BF16 for modern Ampere/Hopper architectures (prevents overflow issues found in FP16)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Configure 4-bit Quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat4 optimized for weight distributions
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True  # Saves additional memory via nested quantization
)

In [ ]:
# ---
# Model & Tokenizer Initialization
# ---
model_id = "google/gemma-2-2b"

# Load model with SDPA (compatible with T4)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

# Enable gradient checkpointing to trade compute for memory
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# Mistral/Llama/Gemma models do not have a pad token by default; use EOS token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Critical for causal LM training

# Apply a standard ChatML template for structural consistency
tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}"

In [ ]:
# ---
# LoRA Configuration
# ---
peft_config = LoraConfig(
    r=16,  # Rank of the update matrices
    lora_alpha=32,  # Scaling factor (usually 2x Rank)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Target all linear layers for maximum expressiveness during adaptation
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

## Data Acquisition & Exploratory Data Analysis (EDA)

Context Block: DPO requires a specialized dataset structure containing a prompt, a chosen response, and a rejected response. Here, we load our proprietary pharmaceutical preference dataset.

---

### The Human Element: Industry-Standard Datasets

SFT lives and dies by the axiom: **Quality over Quantity.** The following Hugging Face datasets are production standards because of their strict formatting, human curation, and conversational depth.

#### `HuggingFaceH4/no_robots`

- **Structure:** Formatted as multi-turn dialogues using the standard `messages` array (`role`, `content`).
- **Why it's used:** It consists exclusively of high-quality, human-written instructions and responses. It explicitly avoids model-generated text, minimizing the ingestion of "AI-isms" or recursive artifacts into the student model.

#### `timdettmers/openassistant-guanaco`

- **Structure:** Formatted with `human`/`assistant` turn delineations.
- **Why it's used:** It serves as a gold standard for multilingual, multi-turn conversational alignment. The dataset relies on aggressive pruning of low-quality responses, teaching the model the exact cadence of helpful, harmless, and honest dialogue.

#### `tatsu-lab/alpaca`

- **Structure:** Tripartite flat structure (`instruction`, `input`, `output`).
- **Why it's used:** Historically foundational. It introduced the concept of **self-instruct** (using a strong teacher model to generate responses for a weaker student to train on). Its structure explicitly separates the system directive (`instruction`) from the contextual data (`input`), which is vital for single-turn task-solving pipelines.


In [ ]:
# ---
# Using HuggingFaceH4/no_robots specifically the SFT split
# ---

dataset = (
    load_dataset("HuggingFaceH4/no_robots", split="train")
    .shuffle(seed=42)
    .select(range(200))
)

def transform_to_standard_format(example):
    """
    Transforms no_robots 'messages' column into the 'prompt'/'completion'
    format expected by modern TRL SFTTrainer.
    """
    messages = example["messages"]

    # Validation: Ensure we have at least a user and assistant turn
    if len(messages) < 2:
        return {"prompt": None, "completion": None}

    # 'prompt' is the conversation up to the last assistant message
    # 'completion' is the last assistant message (as a single-item list)
    prompt = messages[:-1]
    completion = [messages[-1]]

    return {
        "prompt": prompt,
        "completion": completion
    }

# 2. Apply the transformation
# We filter to ensure no empty rows if the dataset had malformed data
dataset = dataset.map(transform_to_standard_format)
dataset = dataset.filter(lambda x: x["prompt"] is not None)

# 3. Clean up the dataset to match the UltraFeedback schema exactly
# This removes unwanted columns
dataset = dataset.remove_columns(["messages", "category", "prompt_id"])

# Inspection:
print(dataset[0])
print(len(dataset))

## Model Development & Training

In [ ]:
training_args = SFTConfig(
    # Output
    output_dir="./output_sft_with_clm",

    # Batch & Gradient
    per_device_train_batch_size=2,
    # per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch = 2 * 4 = 8
    average_tokens_across_devices=True,  # Stable loss across GPUs in DDP

    # Precision
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),  # Use BF16 if supported (Ampere+)

    # Optimizer
    optim="paged_adamw_8bit",  # More memory efficient than 32bit for QLoRA
    adam_beta1=0.9,  # Default, but explicit for reproducibility
    adam_beta2=0.999,  # Default, but explicit for reproducibility
    adam_epsilon=1e-8,
    weight_decay=0.001,
    max_grad_norm=0.3,  # Gradient clipping

    # Learning Rate & Scheduler
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,  # total steps used for warmup
    warmup_steps=0,  # Set to 0 when using warmup_ratio

    # Training Duration
    max_steps=-1,
    num_train_epochs=2,  # Ignored when max_steps > 0, but good to set

    # Sequence & Packing
    max_length=2048,  # critical for memory control
    truncation_mode="keep_start",  # Keep beginning of sequence if truncated
    packing=False,  # Must be False for completion_only_loss
    completion_only_loss=True,  # Only compute loss on assistant response

    # Memory Optimization
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False  # Recommended for newer PyTorch versions
    },
    torch_empty_cache_steps=50,  # Periodically frees CUDA cache to avoid memory fragmentation

    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,  # Only keep last x checkpoints

    # Logging
    logging_strategy="steps",
    logging_steps=5,
    logging_first_step=True,  # Log metrics on step 1
    report_to="none",  # Change to "wandb" or "tensorboard" in production

    # Dataset
    dataset_num_proc=4,  # Parallel dataset processing
    dataset_kwargs={
        "add_special_tokens": False,  # Chat template handles special tokens
        "skip_prepare_dataset": False,
    },

    # Reproducibility
    seed=42,
    data_seed=42,   #  Seed for dataset shuffling
    shuffle_dataset=True,   # Shuffle training data

    # Performance
    dataloader_num_workers=4,  # Parallel data loading
    dataloader_pin_memory=True,  # Faster GPU data transfer
    dataloader_prefetch_factor=2,  # Prefetch 2 batches per worker
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

In [ ]:
print("Initiating SFT Sequence...")
trainer.train()

# Save final adapter weights
trainer.model.save_pretrained("./sft_with_clm_adapter")

### Download Fine-Tuned Adapter
Run the following cell to zip your adapter weights and download them to your local computer.

In [ ]:
import shutil
from google.colab import files

# Name of the folder you want to download
folder_to_zip = './sft_with_clm_adapter'
# Name of the resulting zip file
output_filename = 'sft_adapter_weights.zip'

# Create the zip archive
shutil.make_archive('sft_adapter_weights', 'zip', folder_to_zip)

# Download the file to your machine
# files.download(output_filename)

# Model Usage

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel

In [ ]:
# Enforce BF16 for modern Ampere/Hopper architectures (prevents overflow issues found in FP16)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Configure 4-bit Quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat4 optimized for weight distributions
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True  # Saves additional memory via nested quantization
)

In [ ]:
model_id = "google/gemma-2-2b"

# Load model with SDPA (compatible with T4)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

# Enable gradient checkpointing to trade compute for memory
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# Mistral/Llama/Gemma models do not have a pad token by default; use EOS token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Critical for causal LM training

# Apply a standard ChatML template for structural consistency
tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}"

In [ ]:
# Point this to the checkpoint directory your SFTTrainer created
adapter_path = "./sft_with_clm_adapter"

print(f"Injecting LoRA adapter from {adapter_path}...")
# Wrap the base model with the PEFT weights
tuned_model = PeftModel.from_pretrained(model, adapter_path)

# Put model in evaluation mode (disables dropout layers for consistent inference)
tuned_model.eval()

In [ ]:
# We format our test prompt using the exact same ChatML structure we trained on
test_messages = [
    {"role": "user", "content": "Explain the concept of gravity as if I am a 5 year old."}
]

# Apply template and convert to PyTorch tensors
# add_generation_prompt=True tells the tokenizer to append the assistant's starting tag
input_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize and move to the same GPU as the model
inputs = tokenizer(input_text, return_tensors="pt").to(tuned_model.device)

## Tuned model response

In [ ]:
print("\nGenerating response...\n")

with torch.no_grad(): # Disable gradient tracking to save VRAM and speed up processing
    generated_ids = tuned_model.generate(
        **inputs,
        max_new_tokens=256,  # Maximum length of the generated response
        temperature=0.7,  # Controls randomness (lower = stricter, higher = creative)
        top_p=0.9,  # Nucleus sampling: only consider the top 90% probability mass
        do_sample=True,  # Enables probabilistic sampling
        repetition_penalty=1.15,  # Penalizes the model for repeating the same words
        pad_token_id=tokenizer.eos_token_id
    )

# Slice the output to exclude the input prompt tokens
response_ids = generated_ids[0][inputs.input_ids.shape[-1]:]

# Decode the raw token IDs back into human-readable text
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

print(f"User: {test_messages[0]['content']}")
print(f"Assistant:\n{response_text}")

## Un-tuned model response

In [ ]:
print("\nGenerating response...\n")

with torch.no_grad(): # Disable gradient tracking to save VRAM and speed up processing
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,  # Maximum length of the generated response
        temperature=0.7,  # Controls randomness (lower = stricter, higher = creative)
        top_p=0.9,  # Nucleus sampling: only consider the top 90% probability mass
        do_sample=True,  # Enables probabilistic sampling
        repetition_penalty=1.15,  # Penalizes the model for repeating the same words
        pad_token_id=tokenizer.eos_token_id
    )

# Slice the output to exclude the input prompt tokens
response_ids = generated_ids[0][inputs.input_ids.shape[-1]:]

# Decode the raw token IDs back into human-readable text
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

print("Un-tuned model")
print(f"User: {test_messages[0]['content']}")
print(f"Assistant:\n{response_text}")